# San Francisco, Starring Parking

Every parking lot mapped in OpenStreetMap, in hot pink. Run `fetch_week1.py day3` first.

## 1 · Settings

In [ ]:
DAY      = 3
PROMPT   = "Polygons"
SLUG     = "polygons-sf-parking"
NOTEBOOK = "day03_polygons_sf_parking.ipynb"
TITLE    = "San Francisco,\nStarring Parking"
CREDITS  = "Parking and streets: © OpenStreetMap contributors. City boundary: US Census Bureau. Mapped lots only, private garages are often missing."

## 2 · Load

In [ ]:
import geopandas as gpd, pandas as pd
from mapkit import poster, publish
from mapkit.paths import PROCESSED

parking = gpd.read_file(PROCESSED / "sf_parking.geojson")
streets = gpd.read_file(PROCESSED / "sf_streets.geojson")
park = gpd.read_file(PROCESSED / "sf_ggpark.geojson")
city = gpd.read_file(PROCESSED / "places_six.geojson").query("city == 'sf'")

## 3 · Analysis, every number on the map comes from here

In [ ]:
UTM = 32610
kinds = parking["parking"].fillna("untagged")
# Street-side strips and lanes are not lots, and underground garages don't take up the surface
lots = parking[~kinds.isin(["street_side", "lane", "underground", "on_kerb", "half_on_kerb", "shoulder"])].copy()
lots["km2"] = lots.to_crs(UTM).area / 1e6
area = lots.km2.sum()
park_km2 = park.to_crs(UTM).area.sum() / 1e6
land_km2 = city.to_crs(UTM).area.sum() / 1e6
print(f"All mapped parking   {len(parking):,} shapes, {parking.to_crs(UTM).area.sum()/1e6:.2f} km2")
print(f"Lots on the map      {len(lots):,} shapes, {area:.2f} km2, {area*247.105:.0f} acres")
print(f"Golden Gate Park     {park_km2:.2f} km2")
print(f"Lots vs the park     {area/park_km2*100:.0f}%")
print(f"Share of city land   {area/land_km2*100:.1f}%")
print(lots.parking.fillna('untagged').value_counts())

## 4 · Map

In [ ]:
fig, ax = poster.page("#0b1115")
x, y = poster.merc([-122.5165, -122.355], [37.704, 37.8125])
poster.fit_frame(ax, -122.5165, 37.704, -122.355, 37.8125)
streets_m = streets.to_crs(3857)
minor = streets_m[streets_m.highway.isin(["residential", "unclassified"])]
major = streets_m[~streets_m.highway.isin(["residential", "unclassified"])]
minor.plot(ax=ax, color="#43545c", lw=0.32, zorder=2)
major.plot(ax=ax, color="#62757c", lw=0.6, zorder=2)
lots_m = lots.to_crs(3857)
lots_m.plot(ax=ax, color=poster.PINK, edgecolor=poster.PINK, lw=2.4, alpha=0.18, zorder=3)
lots_m.plot(ax=ax, color=poster.PINK, edgecolor=poster.PINK, lw=0.35, zorder=4)
ax.set_axis_off()
poster.fit_frame(ax, -122.5165, 37.704, -122.355, 37.8125)
poster.title(fig, TITLE, size=40, x=0.06, y=0.955,
             subtitle="The lots on this map add up to\nabout half of Golden Gate Park.", sub_size=15, sub_color="#c6d2d6")
poster.credit(fig, CREDITS)

## 5 · Export

In [ ]:
publish.save(fig, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE.replace("\n", " "),
             subtitle=f"Every mapped parking lot in the city", series="city", tool="Python",
             notebook=NOTEBOOK, credits=CREDITS)